# AIops - Assignment 1
## Question 4 - reproducability

## Step 0 — Setup

In [1]:
# !pip install -r requirements.txt

import json
import pathlib
import random
import subprocess
import sys

import numpy as np
import sklearn
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

# mlflow.set_tracking_uri("http://localhost:5000")
# mlflow.set_experiment("mnist-mlp-repro")
mlflow.set_tracking_uri("http://192.168.0.110:5000")
mlflow.set_experiment("mnist-mlp-repro-partnerB")
client = MlflowClient()

MODEL_NAME = "mnist-mlp"
DATA_FILE = pathlib.Path("data/mnist.npz")

# MLPClassifier stores an AdamOptimizer, which is not on skops' default allowlist.
SKOPS_TRUSTED = ["sklearn.neural_network._stochastic_optimizers.AdamOptimizer"]

print("Tracking URI:", mlflow.get_tracking_uri())

/home/kiran/Desktop/AI_Ops_Assignments/AIops-A1Q4/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tracking URI: http://192.168.0.110:5000


### Pin the seed and capture the git commit

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)


def git_commit():
    """Full SHA of HEAD, or 'unknown' if we are not inside a git repo."""
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        return "unknown"


COMMIT = git_commit()
print("seed       :", SEED)
print("git_commit :", COMMIT)

seed       : 42
git_commit : daa187521e0f00dc7b3e4b19ef48e85c8ffb883e


## Load the DVC-versioned dataset

In [3]:
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"{DATA_FILE} not found. Run `dvc pull` to fetch the versioned dataset "
        "(or `python prepare_data.py` if you are Partner A creating it for the first time)."
    )

d = np.load(DATA_FILE)
X = d["X"].astype(np.float64) / 255.0   # scale to [0, 1] — MLPs are sensitive to input scale
y = d["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"train={X_train.shape}  test={X_test.shape}  classes={len(np.unique(y))}")

train=(56000, 784)  test=(14000, 784)  classes=10


## Train and log the run


In [4]:
HIDDEN = (128, 128)
LEARNING_RATE = 1e-3
ALPHA = 1e-4
MAX_ITER = 100

with mlflow.start_run(run_name="mlp-best-lr0.001-w128-d2") as run:
    # provenance tags
    mlflow.set_tag("git_commit", COMMIT)
    mlflow.set_tag("team", "data-science")
    mlflow.set_tag("python_version", sys.version.split()[0])
    mlflow.set_tag("sklearn_version", sklearn.__version__)
    mlflow.set_tag("numpy_version", np.__version__)

    # parameters
    mlflow.log_param("seed", SEED)
    mlflow.log_param("hidden_layer_sizes", str(HIDDEN))
    mlflow.log_param("width", HIDDEN[0])
    mlflow.log_param("depth", len(HIDDEN))
    mlflow.log_param("learning_rate_init", LEARNING_RATE)
    mlflow.log_param("alpha", ALPHA)
    mlflow.log_param("max_iter", MAX_ITER)
    mlflow.log_param("solver", "adam")
    mlflow.log_param("early_stopping", True)

    # train
    model = MLPClassifier(
        hidden_layer_sizes=HIDDEN,
        alpha=ALPHA,
        learning_rate_init=LEARNING_RATE,
        max_iter=MAX_ITER,
        early_stopping=True,
        n_iter_no_change=5,
        random_state=SEED,
    )
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")

    # metrics
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_macro", f1)
    mlflow.log_metric("n_iter_actual", model.n_iter_)

    for epoch, loss in enumerate(model.loss_curve_):
        mlflow.log_metric("train_loss", loss, step=epoch)
    for epoch, score in enumerate(model.validation_scores_):
        mlflow.log_metric("val_accuracy", score, step=epoch)
    mlflow.log_metric("best_val_score", model.best_validation_score_)

    # log the environment
    pathlib.Path("run_environment.json").write_text(json.dumps({
        "python": sys.version.split()[0],
        "sklearn": sklearn.__version__,
        "numpy": np.__version__,
        "mlflow": mlflow.__version__,
        "git_commit": COMMIT,
        "seed": SEED,
    }, indent=2))
    mlflow.log_artifact("run_environment.json")

    # log the model
    sample_inputs = X_train[:5]
    signature = infer_signature(sample_inputs, model.predict(sample_inputs))

    mlflow.sklearn.log_model(
        model,
        name="model",
        signature=signature,
        input_example=sample_inputs,
        skops_trusted_types=SKOPS_TRUSTED,
    )

    run_id = run.info.run_id

print(f"\nrun_id     = {run_id}")
print(f"accuracy   = {acc:.4f}")
print(f"f1_macro   = {f1:.4f}")
print(f"epochs     = {model.n_iter_}")
print(f"git_commit = {COMMIT}")

🏃 View run mlp-best-lr0.001-w128-d2 at: http://192.168.0.110:5000/#/experiments/3/runs/473ce4225d204c799fadb5bf2d164f29
🧪 View experiment at: http://192.168.0.110:5000/#/experiments/3

run_id     = 473ce4225d204c799fadb5bf2d164f29
accuracy   = 0.9771
f1_macro   = 0.9770
epochs     = 16
git_commit = daa187521e0f00dc7b3e4b19ef48e85c8ffb883e


### Confirm the signature was recorded

In [5]:
from mlflow.models import get_model_info

model_uri = f"runs:/{run_id}/model"
print(get_model_info(model_uri).signature)

inputs: 
  [Tensor('float64', (-1, 784))]
outputs: 
  [Tensor('int64', (-1,))]
params: 
  None



## Register the model and transition it to Staging

In [ ]:
result = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)
print(f"Registered {result.name} version {result.version}")

client.transition_model_version_stage(
    name=MODEL_NAME, version=result.version, stage="Staging",
)
print(f"Version {result.version} -> Staging")

client.set_model_version_tag(MODEL_NAME, result.version, "git_commit", COMMIT)
client.set_model_version_tag(MODEL_NAME, result.version, "run_id", run_id)
client.update_model_version(
    name=MODEL_NAME,
    version=result.version,
    description=(
        f"Best sweep config: lr={LEARNING_RATE}, hidden={HIDDEN}, seed={SEED}. "
        f"accuracy={acc:.4f}, f1_macro={f1:.4f}, converged in {model.n_iter_} epochs. "
        f"Code at git {COMMIT[:8]}."
    ),
)
print("Tags and description set.")

## Load the model back and verify


In [6]:
import mlflow.pyfunc

# Kevin's registered versions point at artifacts on his local disk (artifact_location = /home/kevin/.../mlruns/1), so they are not retrievable over the network. 
loaded = mlflow.pyfunc.load_model(f"runs:/{run_id}/model")
print("Predictions from the registered Staging model:", loaded.predict(X_test[:8]))
print("Ground truth                                 :", y_test[:8])

Predictions from the registered Staging model: [7 3 1 1 2 5 9 3]
Ground truth                                 : [7 3 1 1 2 5 9 8]


In [7]:
for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
    full = client.get_model_version(MODEL_NAME, v.version)
    print(f"v{v.version:>2}  stage={v.current_stage:<10}  run_id={v.run_id}  "
          f"git_commit={full.tags.get('git_commit', '')[:8]}")

v 2  stage=Staging     run_id=0875f2e1bcd640579e22769eff843811  git_commit=daa18752
v 1  stage=Staging     run_id=08886790b99a4cb5b8f07fcacf15a1a6  git_commit=daa18752


In [8]:
# Partner A, run 661367a0, accuracy = 0.9771428571428571

REFERENCE_ACCURACY = 0.9771
TOLERANCE = 0.005

delta = acc - REFERENCE_ACCURACY
matched = abs(delta) <= TOLERANCE
verdict = "MATCHED" if matched else "MISMATCH"

note = (
    f"Reproduction verdict: {verdict}\n"
    f"Partner A accuracy : {REFERENCE_ACCURACY:.4f}\n"
    f"Partner B accuracy : {acc:.4f}\n"
    f"Difference         : {delta:+.4f}  (tolerance +/-{TOLERANCE})\n"
    f"git_commit         : {COMMIT[:8]}\n"
    f"python / sklearn   : {sys.version.split()[0]} / {sklearn.__version__}\n"
)

if matched:
    note += (
        "The metric reproduced within tolerance. The pinned seed, the "
        "DVC-versioned dataset and the pinned requirements.txt were sufficient.\n"
    )

client.set_tag(run_id, "reproduction_verdict", verdict)
client.set_tag(run_id, "reproduction_delta", f"{delta:+.4f}")
client.log_text(run_id, note, "reproduction_note.txt")

print(note)
print(f"Logged to run {run_id}")


Reproduction verdict: MATCHED
Partner A accuracy : 0.9771
Partner B accuracy : 0.9771
Difference         : +0.0000  (tolerance +/-0.005)
git_commit         : daa18752
python / sklearn   : 3.14.4 / 1.9.0
The metric reproduced within tolerance. The pinned seed, the DVC-versioned dataset and the pinned requirements.txt were sufficient.

Logged to run 473ce4225d204c799fadb5bf2d164f29
